# <font color="#418FDE" size="6.5" uppercase>**Feature Expansion**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Discretize continuous variables using thresholding and binning strategies suited to the modeling goal. 
- Create polynomial, interaction, and spline features to represent nonlinear relationships. 
- Prevent preprocessing leakage by fitting transformations only on training data or inside pipelines. 


## **1. Discretization Methods**

### **1.1. Binning Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_01.jpg?v=1783832868" width="250">



>* Binning groups continuous values into ordered ranges.
>* It highlights meaningful patterns over small differences.

>* Cut points assign values into ordered ranges.
>* Binning simplifies noisy data and highlights thresholds.

>* Binning simplifies data but loses detail.
>* Choose bins based on purpose and meaning.



In [ ]:
#@title Python Code - Binning Basics

# This example introduces simple numeric binning.
# We group values into ordered intervals.
# The output shows labels and counts.

import numpy as np
import pandas as pd

# Create a small continuous feature.
ages = np.array([18, 22, 27, 31, 36, 42, 49, 55])

# Define clear bin boundaries.
bin_edges = [0, 25, 40, 60]
bin_names = ["young", "adult", "mature"]

# Convert numbers into ordered categories.
age_groups = pd.cut(
    ages, bins=bin_edges, labels=bin_names,
    right=False, include_lowest=True
)

# Build a tiny summary table.
summary = pd.DataFrame(
    {"age": ages, "age_group": age_groups}
)

# Count how many values land per bin.
counts = summary["age_group"].value_counts().sort_index()

# Show the original values.
print("Ages:", ages.tolist())

# Show each age with its bin.
print(summary.to_string(index=False))

# Show the ordered bin counts.
print("Bin counts:", counts.to_dict())



### **1.2. Binning Strategies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_02.jpg?v=1783832870" width="250">



>* Choose bins based on data and goals.
>* Equal-frequency handles skew better than equal-width.

>* Domain-based bins improve interpretability and relevance.
>* Poor thresholds can distort similar values.

>* Balance interpretability, stability, and predictive value.
>* Compare bin choices to fit purpose.



In [ ]:
#@title Python Code - Binning Strategies

# This example compares common binning strategies.
# It uses a small skewed numeric sample.
# The output highlights interpretability and balance.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set a deterministic seed.
rng = np.random.default_rng(7)

# Create a small skewed feature.
income_k = np.round(rng.lognormal(3.2, 0.55, 40), 1)

data = pd.DataFrame({"income_k": income_k})

# Build equal-width bins.
data["equal_width"] = pd.cut(
    data["income_k"], bins=4, include_lowest=True
)

# Build equal-frequency bins.
data["equal_freq"] = pd.qcut(
    data["income_k"], q=4, duplicates="drop"
)

# Build domain-based bins.
domain_edges = [0, 20, 30, 45, np.inf]
domain_labels = ["low", "mid", "upper_mid", "high"]

data["domain_bins"] = pd.cut(
    data["income_k"], bins=domain_edges, labels=domain_labels
)

# Check the sample size.
assert len(data) == 40, "Unexpected sample size."

# Count observations per strategy.
width_counts = data["equal_width"].value_counts().sort_index()
freq_counts = data["equal_freq"].value_counts().sort_index()

domain_counts = data["domain_bins"].value_counts().sort_index()

# Print a compact comparison.
print("Income sample size:", len(data))
print("Min and max:", float(data.income_k.min()), float(data.income_k.max()))
print("Equal-width counts:", width_counts.to_dict())
print("Equal-frequency counts:", freq_counts.to_dict())
print("Domain-based counts:", domain_counts.to_dict())

# Plot the original values and thresholds.
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(data["income_k"], bins=10, color="skyblue", edgecolor="black")

# Add domain cut lines.
for edge in domain_edges[1:-1]:
    ax.axvline(edge, color="crimson", linestyle="--", linewidth=1.5)

# Label the teaching plot.
ax.set_title("Binning strategies start from the same continuous feature")
ax.set_xlabel("Income in thousands of dollars")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()



### **1.3. Binning Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_03.jpg?v=1783832872" width="250">



>* Binning groups continuous values into ordered ranges.
>* Ranges improve interpretation while reducing unnecessary precision.

>* Bins group similar values for analysis.
>* Simplifies noise but loses fine detail.

>* Choose bins to match goals and meaning.
>* Balance simplicity, detail, and decision usefulness.



In [ ]:
#@title Python Code - Binning Basics

# This example introduces simple numeric binning.
# We group ages into meaningful ranges.
# Bins trade detail for clearer categories.

import numpy as np
import pandas as pd

# Create a tiny continuous feature.
ages = np.array([18, 22, 29, 34, 41, 53, 67])

# Define ordered bin edges.
bin_edges = [0, 25, 40, 60, 100]

# Define readable bin labels.
bin_labels = ["young", "adult", "middle", "senior"]

# Convert ages into categories.
age_groups = pd.cut(
    ages, bins=bin_edges, labels=bin_labels,
    right=False, include_lowest=True
)

# Build a small teaching table.
summary = pd.DataFrame(
    {"age": ages, "age_group": age_groups.astype(str)}
)

# Show the original and binned values.
print(summary.to_string(index=False))

# Count how many values land per bin.
print("\nCounts by bin:")
print(summary["age_group"].value_counts().sort_index().to_string())



## **2. Nonlinear Feature Bases**

### **2.1. Polynomial Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_01.jpg?v=1783832863" width="250">



>* Polynomial features model curved, nonlinear relationships.
>* Squared and cubed terms add flexibility.

>* Higher powers capture different nonlinear curve shapes.
>* Useful for turning points and diminishing returns.

>* Higher degrees risk overfitting and instability.
>* Use simple validated expansions with careful scaling.



In [ ]:
#@title Python Code - Polynomial Features

# Polynomial features can model simple curved relationships.
# This example builds powers of one feature.
# A plot compares linear and curved fits.

import numpy as np
import matplotlib.pyplot as plt

# Make a tiny deterministic dataset.
rng = np.random.default_rng(7)
x = np.linspace(0.0, 10.0, 25)
noise = rng.normal(0.0, 2.0, size=x.shape)
y = 3.0 + 2.0 * x - 0.25 * x**2 + noise

# Validate matching input lengths.
assert x.shape == y.shape
x_col = x.reshape(-1, 1)

# Build linear and polynomial design matrices.
X_linear = np.column_stack((np.ones_like(x), x))
X_poly = np.column_stack((np.ones_like(x), x, x**2))

# Fit both models with least squares.
coef_linear = np.linalg.lstsq(X_linear, y, rcond=None)[0]
coef_poly = np.linalg.lstsq(X_poly, y, rcond=None)[0]

# Predict on a smooth grid.
x_plot = np.linspace(x.min(), x.max(), 200)
Y_linear = coef_linear[0] + coef_linear[1] * x_plot
Y_poly = coef_poly[0] + coef_poly[1] * x_plot + coef_poly[2] * x_plot**2

# Show the learned equations.
print(f"Linear fit: y = {coef_linear[0]:.2f} + {coef_linear[1]:.2f}x")
print(f"Polynomial fit: y = {coef_poly[0]:.2f} + {coef_poly[1]:.2f}x + {coef_poly[2]:.2f}x^2")

# Plot data and both fitted curves.
plt.figure(figsize=(7, 4))
plt.scatter(x, y, color="black", label="Observed data")
plt.plot(x_plot, Y_linear, label="Linear fit", linewidth=2)
plt.plot(x_plot, Y_poly, label="Polynomial fit", linewidth=2)

# Label the chart clearly.
plt.xlabel("Study hours")
plt.ylabel("Exam score")
plt.title("Polynomial features capture curvature")
plt.legend()

# Display the final teaching plot.
plt.tight_layout()
plt.show()



### **2.2. Interaction Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_02.jpg?v=1783832866" width="250">



>* Interactions model features affecting each other.
>* They make simple models capture conditional patterns.

>* Interactions capture effects from combined conditions.
>* They model context-specific patterns across groups.

>* Interactions add power but risk overfitting.
>* Choose carefully and scale variables sensibly.



In [ ]:
#@title Python Code - Interaction Features

# Interaction features capture combined variable effects.
# This example builds simple pairwise products.
# Small arrays keep the idea beginner friendly.

import numpy as np
import pandas as pd

# Create two original numeric features.
study_hours = np.array([1, 2, 3, 4, 5])
prior_score = np.array([50, 55, 60, 65, 70])
interaction = study_hours * prior_score

# Check matching lengths before combining.
assert study_hours.shape == prior_score.shape
assert interaction.size == study_hours.size

# Build a tiny teaching table.
data = pd.DataFrame({
    "study_hours": study_hours,
    "prior_score": prior_score,
    "hours_x_score": interaction,
})

# Show how the new feature changes.
print("Original and interaction features:")
print(data.to_string(index=False))

# Compare additive and interaction signals.
additive = study_hours + prior_score
print("\nAdditive values:", additive.tolist())
print("Interaction values:", interaction.tolist())



### **2.3. Spline Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_03.jpg?v=1783832865" width="250">



>* Splines model smooth bends across variable ranges.
>* Derived features capture local nonlinear changes.

>* Knots guide smooth changes across regions.
>* Splines balance flexibility better than binning.

>* Splines model curved patterns without one shape.
>* Choose knots carefully to balance flexibility.



In [ ]:
#@title Python Code - Spline Features

# This script demonstrates smooth spline basis features.
# We compare linear and spline curve fits.
# The example uses tiny synthetic data.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import SplineTransformer

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create one numeric feature.
x = np.linspace(0.0, 10.0, 80).reshape(-1, 1)

# Create a smooth nonlinear target.
y = np.sin(x[:, 0]) + 0.15 * rng.normal(size=x.shape[0])

# Build spline basis features.
spline = SplineTransformer(n_knots=5, degree=3, include_bias=False)

# Fit only on the feature values.
X_spline = spline.fit_transform(x)

# Check the transformed shape.
assert X_spline.shape[0] == x.shape[0]

# Add an intercept column.
X_design = np.column_stack([np.ones(len(x)), X_spline])

# Solve a small linear regression.
coef, _, _, _ = np.linalg.lstsq(X_design, y, rcond=None)

# Compute fitted values.
y_hat = X_design @ coef

# Show a short explanation.
print("Original feature shape:", x.shape)
print("Spline feature shape:", X_spline.shape)
print("Spline features create smooth local flexibility.")

# Plot data and fitted spline curve.
plt.figure(figsize=(8, 4))
plt.scatter(x[:, 0], y, s=22, alpha=0.7, label="Observed data")
plt.plot(x[:, 0], y_hat, color="crimson", linewidth=2.5, label="Spline fit")
plt.xlabel("Input feature")
plt.ylabel("Target")
plt.title("Spline Features Capture Nonlinear Relationships")

# Finish the single teaching plot.
plt.legend(); plt.tight_layout(); plt.show()



## **3. Leakage Safe Transforms**

### **3.1. Fit On Training**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_01.jpg?v=1783832874" width="250">



>* Learn transforms only from training data.
>* Apply fixed rules to validation and test.

>* Full-data binning leaks evaluation information.
>* Learn bins from training, reuse unchanged.

>* Training-only fitting ensures fair, trustworthy evaluation.
>* Refitting on test data leaks future information.



In [ ]:
#@title Python Code - Fit On Training

# Fit transforms using training data only.
# This example shows leakage safe binning.
# Small outputs keep the lesson clear.

import numpy as np
import pandas as pd

# Make results deterministic and simple.
rng = np.random.default_rng(7)
size = 24

# Create one continuous feature.
income = rng.normal(60, 18, size).clip(20, 110)
target = (income > 62).astype(int)

# Build a tiny labeled table.
data = pd.DataFrame({"income": income, "target": target})
train = data.iloc[:16].copy()

test = data.iloc[16:].copy()
assert len(train) > 0 and len(test) > 0

# Learn bin edges from training only.
train_edges = np.quantile(train["income"], [0.0, 0.5, 1.0])
train_edges[0] -= 1e-9

# Apply fixed training edges everywhere.
train_bins = pd.cut(train["income"], bins=train_edges, labels=["low", "high"])
test_bins_safe = pd.cut(test["income"], bins=train_edges, labels=["low", "high"])

# Incorrectly learn edges from full data.
full_edges = np.quantile(data["income"], [0.0, 0.5, 1.0])
full_edges[0] -= 1e-9

# Compare safe and leaky test transforms.
test_bins_leaky = pd.cut(test["income"], bins=full_edges, labels=["low", "high"])
changed = int((test_bins_safe != test_bins_leaky).sum())

# Print a short teaching summary.
print("Train median edge:", round(train_edges[1], 2))
print("Full median edge:", round(full_edges[1], 2))
print("Test rows changed by leakage:", changed)

# Show a few transformed test rows.
preview = pd.DataFrame({
    "income": test["income"].round(1),
    "safe_bin": test_bins_safe.astype(str),
    "leaky_bin": test_bins_leaky.astype(str),
})
print(preview.head(5).to_string(index=False))



### **3.2. Pipeline Fitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_02.jpg?v=1783832876" width="250">



>* Fit all preprocessing only on training data.
>* Pipelines prevent leakage and inflated performance estimates.

>* Full-data preprocessing leaks future information.
>* Pipelines keep evaluation realistic and honest.

>* Pipelines preserve consistent preprocessing for new data.
>* They improve reproducibility and honest performance estimates.



### **3.3. Pipeline Fit Only**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_03.jpg?v=1783832878" width="250">



>* Pipelines learn transforms from training data only.
>* Reuse unchanged transforms for fair evaluation.

>* Train-only pipelines mirror real deployment conditions.
>* They prevent optimistic leakage across domains.

>* Fit preprocessing separately within each cross-validation fold.
>* Pipelines prevent leakage and improve reproducibility.



In [ ]:
#@title Python Code - Pipeline Fit Only

# Pipelines prevent leakage during preprocessing.
# Training data must define transformations.
# This example compares safe workflows.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer, PolynomialFeatures

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create a tiny nonlinear dataset.
x = np.linspace(10, 100, 40)
noise = rng.normal(0, 8, 40)

y = 0.08 * x + 0.015 * (x - 55) ** 2 + noise

df = pd.DataFrame({"income_k": x, "risk": y})

# Split before fitting safe transforms.
X = df[["income_k"]]
y = df["risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=7
)

# Fit a leaky transform on all data.
leaky_bins = KBinsDiscretizer(
    n_bins=4, encode="onehot-dense", strategy="quantile"
)

X_all_binned = leaky_bins.fit_transform(X)
X_train_leaky = X_all_binned[X_train.index]
X_test_leaky = X_all_binned[X_test.index]

# Add polynomial terms after leaky binning.
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_leaky = poly.fit_transform(X_train_leaky)
X_test_leaky = poly.transform(X_test_leaky)

# Train a model on leaky features.
leaky_model = LinearRegression()
leaky_model.fit(X_train_leaky, y_train)

leaky_pred = leaky_model.predict(X_test_leaky)
leaky_rmse = mean_squared_error(
    y_test, leaky_pred, squared=False
)

# Build a safe pipeline.
safe_pipe = Pipeline([
    (
        "bin",
        KBinsDiscretizer(
            n_bins=4, encode="onehot-dense", strategy="quantile"
        ),
    ),
    (
        "poly",
        PolynomialFeatures(degree=2, include_bias=False),
    ),
    ("model", LinearRegression()),
])

# Fit only on training data.
safe_pipe.fit(X_train, y_train)
safe_pred = safe_pipe.predict(X_test)

safe_rmse = mean_squared_error(
    y_test, safe_pred, squared=False
)

# Inspect learned bin edges.
leaky_edges = np.round(leaky_bins.bin_edges_[0], 1)
safe_edges = np.round(
    safe_pipe.named_steps["bin"].bin_edges_[0], 1
)

# Print a short teaching summary.
print("Train size:", len(X_train), "Test size:", len(X_test))
print("Leaky bin edges:", leaky_edges.tolist())
print("Safe bin edges:", safe_edges.tolist())
print("Leaky test RMSE:", round(leaky_rmse, 2))
print("Safe test RMSE:", round(safe_rmse, 2))
print("Lesson: fit transforms after the split.")



# <font color="#418FDE" size="6.5" uppercase>**Feature Expansion**</font>


In this lecture, you learned to:
- Discretize continuous variables using thresholding and binning strategies suited to the modeling goal. 
- Create polynomial, interaction, and spline features to represent nonlinear relationships. 
- Prevent preprocessing leakage by fitting transformations only on training data or inside pipelines. 

In the next Module (Module 5), we will go over 'None'